# 06_checkpoint_eval

对应 `index.md` 第 6 阶段：`state_dict`、保存加载、`train` / `eval`。

笔记本在 `codes/pytorch/06_checkpoint_eval/qa.ipynb`。需要读写文件时，工作空间就是这个目录。

先运行下一格，得到 `ROOT`。每题只改 `# 作答` 下面的代码。前置代码不用改。做完自己跑通即可，先不要对答案。


In [9]:
from pathlib import Path

def lab_root() -> Path:
    """qa.ipynb 所在目录。"""
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if folder.name == "06_checkpoint_eval" and (folder / "qa.ipynb").is_file():
            return folder
        candidate = folder / "codes" / "pytorch" / "06_checkpoint_eval"
        if (candidate / "qa.ipynb").is_file():
            return candidate
    return here

ROOT = lab_root()
ROOT


PosixPath('/Users/keyficller/Documents/AEFS-Notes/codes/pytorch/06_checkpoint_eval')

## 1. 看 state_dict 的键

已有 `net`。取出它的 `state_dict`，打印所有键名（可用 `list(...keys())`）。


In [10]:
import torch
from torch import nn

net = nn.Sequential(nn.Linear(3, 4), nn.ReLU(), nn.Linear(4, 1))

# 作答

state_dict = net.state_dict()

print(list(state_dict.keys()))
#评阅
# 对。Sequential 里第 0、2 层 Linear 的 weight/bias 键都列出来了。

#参考答案
# print(list(net.state_dict().keys()))

['0.weight', '0.bias', '2.weight', '2.bias']


## 2. 保存权重到文件

把 `net` 的 `state_dict` 保存到 `ROOT / "q2" / "net.pt"`（目录不存在就先创建）。

保存后打印该文件是否存在。


In [ ]:
import torch
from torch import nn
from pathlib import Path

q2 = ROOT / "q2"
q2.mkdir(parents=True, exist_ok=True)
ckpt_path = q2 / "net.pt"
if ckpt_path.exists():
    ckpt_path.unlink()

torch.manual_seed(0)
net = nn.Linear(2, 1)

# 作答

torch.save(net.state_dict(), ckpt_path)

print(ckpt_path.exists())
#评阅
# 目标路径 q2/net.pt 有文件。但作答用了 path，前置定义的是 ckpt_path，应对齐其中一个；mkdir 本可放在作答里。

#参考答案
# path = ROOT / "q2" / "net.pt"
# path.parent.mkdir(parents=True, exist_ok=True)
# torch.save(net.state_dict(), path)
# print(path.exists())

True


## 3. 加载到新模型

已有结构相同的空模型 `new_net`，以及权重文件 `ckpt_path`（在 `ROOT / "q3" / "linear.pt"`）。

加载权重进 `new_net`。用同一份 `x`，打印 `net(x)` 与 `new_net(x)` 是否 `allclose`。


In [14]:
import torch
from torch import nn
from pathlib import Path

q3 = ROOT / "q3"
q3.mkdir(parents=True, exist_ok=True)
ckpt_path = q3 / "linear.pt"

torch.manual_seed(0)
net = nn.Linear(2, 1)
torch.save(net.state_dict(), ckpt_path)

new_net = nn.Linear(2, 1)
x = torch.randn(3, 2)

# 作答

new_net.load_state_dict(torch.load(ckpt_path))
print(torch.allclose(net(x), new_net(x)))
#评阅
# 对。load_state_dict 后与原 net 输出一致。

#参考答案
# new_net.load_state_dict(torch.load(ckpt_path))
# print(torch.allclose(net(x), new_net(x)))

True


## 4. 保存更完整的 checkpoint

把一个 dict 存到 `ROOT / "q4" / "train.pt"`（目录不存在就先创建），至少包含：

- `"model"`: `net.state_dict()`
- `"optim"`: `opt.state_dict()`
- `"epoch"`: 整数 `3`

再读回来，打印读出的 `epoch`。


In [16]:
import torch
from torch import nn
from pathlib import Path

q4 = ROOT / "q4"
q4.mkdir(parents=True, exist_ok=True)
ckpt_path = q4 / "train.pt"
if ckpt_path.exists():
    ckpt_path.unlink()

torch.manual_seed(0)
net = nn.Linear(2, 1)
opt = torch.optim.SGD(net.parameters(), lr=0.01)

# 作答

torch.save({
    "model": net.state_dict(),  
    "optim": opt.state_dict(),
    "epoch": 3,
}, ckpt_path)

print(torch.load(ckpt_path)["epoch"])
#评阅
# 对。dict 含 model/optim/epoch，读回 epoch 为 3。

#参考答案
# path = ROOT / "q4" / "train.pt"
# path.parent.mkdir(parents=True, exist_ok=True)
# torch.save({"model": net.state_dict(), "optim": opt.state_dict(), "epoch": 3}, path)
# print(torch.load(path)["epoch"])

3


## 5. eval 推理时关掉梯度

已有带 Dropout 的 `net`。切到 `eval`，在**不追踪梯度**的情况下对 `x` 前向两次。

打印：两次输出是否 `allclose`；以及第二次输出的 `requires_grad`。


In [18]:
import torch
from torch import nn

torch.manual_seed(0)
net = nn.Sequential(nn.Linear(3, 3), nn.Dropout(0.5), nn.Linear(3, 1))
x = torch.randn(2, 3)

# 作答

net.eval()

first, second = net(x), net(x)
print(torch.allclose(first, second))
print(second.requires_grad)
#评阅
# eval 两次一致对了，但缺「不追踪梯度」：应用 with torch.no_grad()，此时 requires_grad 应为 False（你打出的是 True）。

#参考答案
# net.eval()
# with torch.no_grad():
#     a, b = net(x), net(x)
# print(torch.allclose(a, b))
# print(b.requires_grad)

True
True
